In [8]:
from pathlib import Path
import vtk
from pathlib import Path
from scipy.stats import multivariate_normal
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy import io as scio
from scipy import stats as sci_stats
import pickle
import scipy.spatial as sp
from typing import Union,Dict,List
import copy
import pickle

In [9]:
path_to_processed_cohort = r"/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/mouse_skull_workdir"
path_to_processed_abnormals = r"/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/mouse_ablormal"
result_energies = r"/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/single_comparisonTrabd2_F_E64-00965.mat"

In [13]:
def compute_center_of_mesh(mesh):
    centerOfMassFilter = vtk.vtkCenterOfMass()
    centerOfMassFilter.SetInputData(mesh)
    centerOfMassFilter.SetUseScalarsAsWeights(False)
    centerOfMassFilter.Update();
    center = centerOfMassFilter.GetCenter()
    return center



reader = vtk.vtkOBJReader()
num_of_points = 0
meshes_dict = {}

for path in tqdm(list(Path(path_to_processed_cohort).iterdir())):
    mesh_file = path / "head_affine_registered.obj"
    if not mesh_file.exists():
        continue
    reader.SetFileName(mesh_file)
    reader.Update()
    mesh = reader.GetOutput()
    points = mesh.GetPoints()
    center_of_mesh = np.array( compute_center_of_mesh(mesh) )
    num_of_points = points.GetNumberOfPoints()
    res_mesh = []
    for i in range(points.GetNumberOfPoints()):
        pt = points.GetPoint(i)
        res_mesh.append(list( np.array( pt) - center_of_mesh ))
    meshes_dict[str( mesh_file.parent.name ) ] = res_mesh

  0%|          | 0/769 [00:00<?, ?it/s]

In [8]:
abnormals = {}

for path in tqdm(list(Path(path_to_processed_abnormals).iterdir())):
    mesh_file = path / "head_affine_registered.obj"
    if not mesh_file.exists():
        continue
    reader.SetFileName(mesh_file)
    reader.Update()
    mesh = reader.GetOutput()
    points = mesh.GetPoints()
    center_of_mesh = np.array( compute_center_of_mesh(mesh) )
    num_of_points = points.GetNumberOfPoints()
    res_mesh = []
    for i in range(points.GetNumberOfPoints()):
        pt = points.GetPoint(i)
        res_mesh.append(list( np.array( pt) - center_of_mesh ))
    abnormals[str( mesh_file.parent.name ) ] = res_mesh
    


  0%|          | 0/97 [00:00<?, ?it/s]

In [14]:
def compute_outliers(meshes_dict: Union[Dict[str,np.ndarray],List[np.ndarray]]):
    
    stats = []
    index_sum=None
    if isinstance(meshes_dict,list):
        num_of_points = len( meshes_dict[0] )
        working_dict = list(zip(range(num_of_points),meshes_dict))
        
    else:
        num_of_points = len( list(meshes_dict.values())[0] )
        working_dict = list(meshes_dict.items())
            
    for i in tqdm(range(num_of_points)): # i in range of num_of_points
        index = []
        points = []
        #print (i, working_dict)
        for key, mesh_v in working_dict:
            index.append(key)
            points.append(mesh_v[i])
        
        df = pd.DataFrame( data = points, index=index, columns=['x','y','z'])
        
        indexes = (np.abs(sci_stats.zscore(df)) < 2).all(axis=1)
        
        if index_sum is None:
            index_sum = (~indexes).astype(int)
        else:
            index_sum += (~indexes).astype(int)
        df = df[indexes]
        stats.append(df.describe())
    
    return stats, index_sum

    

stats, index_sum = compute_outliers(meshes_dict)
meshes_dict_copy = copy.copy(meshes_dict)


#copy dict 
# for k,v in sorted(index_sum.items(), key=lambda x:x[1]):
#     if v > 1000:
#         meshes_dict_copy.pop(k)

  0%|          | 0/25680 [00:00<?, ?it/s]

In [10]:
def procreustes_alignment(reference_mesh, compute_mesh):
    #("STARTED")
    matx1,mesh_aligned, translation = sp.procrustes(reference_mesh, compute_mesh)

    return mesh_aligned

def gpa(meshes: Dict[str, np.ndarray], max_iterations=4, threshold=1e-6):
    num_meshes = len(meshes)
    num_vertices = len(list(meshes.values())[0])

    # Convert meshes to 3D numpy array
    stacked_meshes = np.stack(meshes.values())
    
    mesh_keys = list(meshes.keys())

    # Initialize reference shape as the first mesh
    reference_mesh = stacked_meshes[0]

    for iteration in tqdm(range(max_iterations)):

        # Step 1: Superimpose all meshes to the current reference shape
        aligned_meshes = [reference_mesh]

        # futures = [executor.submit(procreustes_alignment, reference_mesh, mesh) for mesh in stacked_meshes[1:]]
        #
        # for f in tqdm(futures):
        #     aligned_meshes.append( f.result() )

        for mesh in tqdm(stacked_meshes[1:]):
            aligned_meshes.append( procreustes_alignment(reference_mesh, mesh) )
        # Step 2: Compute the mean shape of the current set of superimposed shapes
        mean_mesh = np.mean(np.stack(aligned_meshes), axis=0)
        #print("mean mesh shape", mean_mesh.shape)
        # Step 3: Check for convergence based on Procrustes distance
        _, _, procrustes_distance = sp.procrustes(reference_mesh, mean_mesh)
        print("procrustes distance ", procrustes_distance)
        if procrustes_distance < threshold:
            break

        # Update the reference shape
        reference_mesh = mean_mesh
        
    result_shapes = {}
    for key, shape in meshes.items():
        result = procreustes_alignment(reference_mesh=reference_mesh,compute_mesh=shape)
        result_shapes[key] = result

    return reference_mesh, aligned_meshes, result_shapes

In [42]:
#procrustes analysis
import copy
ref_mesh, aligned_meshes, shapes= gpa(meshes_dict_copy,max_iterations=100, threshold=1e-11)
#divide male to female
males = { k: v for k,v in shapes.items() if k.split("_")[1] == "M" }
females = { k: v for k,v in shapes.items() if k.split("_")[1] == "F" }

/usr/local/lib/python3.8/dist-packages/IPython/core/interactiveshell.py:3398: FutureWarning: arrays to stack must be passed as a "sequence" type such as list or tuple. Support for non-sequence iterables such as generators is deprecated as of NumPy 1.16 and will raise an error in the future.
  exec(code_obj, self.user_global_ns, self.user_ns)


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/477 [00:00<?, ?it/s]

procrustes distance  3.982895395549866e-05


  0%|          | 0/477 [00:00<?, ?it/s]

procrustes distance  0.00022094432283084397


  0%|          | 0/477 [00:00<?, ?it/s]

procrustes distance  1.109844319598831e-08


  0%|          | 0/477 [00:00<?, ?it/s]

procrustes distance  5.1463720368632115e-14


In [43]:
females_and_trapbd = copy.copy(females)
females_and_trapbd.update( { "Trabd2_F_E64-00965" : abnormals["Trabd2_F_E64-00965"] } )
_,_,res_meshes = gpa(females_and_trapbd ,max_iterations=100, threshold=1e-11)

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/221 [00:00<?, ?it/s]

procrustes distance  0.0003391501293304714


  0%|          | 0/221 [00:00<?, ?it/s]

procrustes distance  7.1496631911983356e-09


  0%|          | 0/221 [00:00<?, ?it/s]

procrustes distance  1.5076535883369644e-13


In [44]:

from sklearn.covariance import EmpiricalCovariance

def single_to_many ( mesh_pts: np.ndarray, cohort : List[np.ndarray]):
    
    # build distributions
    num_of_points = mesh_pts.shape[0]
    dists = []
    for i in tqdm(range(num_of_points)): # i in range of num_of_points
        points = []
        #print (i, working_dict)
        
        for mesh_v in cohort:
            points.append(mesh_v[i])
            
        points = np.stack(points)
        
        emp_cov = EmpiricalCovariance().fit(points)
        
        mahalanobis_distance = emp_cov.mahalanobis([mesh_pts[i]]) [0]
        
        dists.append(mahalanobis_distance)
            
    return dists
    
    

In [45]:
art = res_meshes["Trabd2_F_E64-00965"]
res_meshes.pop("Trabd2_F_E64-00965")
dists = single_to_many ( mesh_pts=art, cohort=list(res_meshes.values() )  )

  0%|          | 0/25680 [00:00<?, ?it/s]

In [46]:
polygons = mesh.GetPolys()
faces = []
# Iterate over the polygons to extract the faces
polygons.InitTraversal()
while True:
    faceIds = vtk.vtkIdList()
    if polygons.GetNextCell(faceIds):
        face = [faceIds.GetId(i) for i in range(faceIds.GetNumberOfIds())]
        faces.append(face)
    else:
        break
        
### compute colors and write to mat 
energies = []
for x in faces:
    en = 0 
    for p in x:
        en+= dists[p]
    en = en / 3 
    energies.append(en )

scio.savemat(result_energies, {"energies": energies})

In [30]:
res_meshes

{'wt_F_82Z-CG092': array([[-0.00130997, -0.00091855, -0.00954387],
        [-0.0014578 , -0.00083332, -0.00955187],
        [ 0.0019558 , -0.00137518, -0.00931048],
        ...,
        [-0.00278264, -0.00335807, -0.00844929],
        [ 0.00057471,  0.00506605,  0.00355397],
        [-0.00385233,  0.00012627, -0.00205261]]),
 'wt_F_82Z-06731': array([[-1.49283700e-03, -1.12859691e-03, -9.57542580e-03],
        [-1.64543826e-03, -1.05338164e-03, -9.58555171e-03],
        [ 1.88768773e-03, -1.46197944e-03, -9.20260369e-03],
        ...,
        [-2.73009581e-03, -3.29203285e-03, -8.45805901e-03],
        [ 5.74070626e-04,  5.27824700e-03,  3.39281846e-03],
        [-3.82257751e-03,  4.14616457e-05, -1.89314610e-03]]),
 'wt_F_82Z-07750': array([[-0.00131371, -0.00061851, -0.00937552],
        [-0.00144736, -0.00054468, -0.00931051],
        [ 0.00201276, -0.00139788, -0.00938099],
        ...,
        [-0.00276543, -0.00351138, -0.00851216],
        [ 0.00074985,  0.00520304,  0.0034983 ]

In [14]:
len(meshes_dict_copy), len(meshes_dict)

(208, 643)

In [18]:
males = { k: v for k,v in meshes_dict_copy.items() if k.split("_")[1] == "M" }
females = { k: v for k,v in meshes_dict_copy.items() if k.split("_")[1] == "F" }
o_f =   "/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/wt_cohorts_pts.pkl"
of = open(o_f,'wb')
pickle.dump({"M": males, "F": females},of)
of.close()

In [15]:
sorted(index_sum.items(), key=lambda x:x[1])

[('wt_M_82Z-10911', 0),
 ('wt_M_82Z-CF279', 0),
 ('wt_M_82Z-02973', 0),
 ('wt_M_82Z-11327', 0),
 ('wt_M_82Z-CG256', 0),
 ('wt_F_82Z-CG092', 0),
 ('wt_M_82Z-08685', 0),
 ('wt_M_82Z-11639', 0),
 ('wt_M_82Z-CH035', 0),
 ('wt_F_82Z-07311', 0),
 ('wt_F_82Z-CH156', 0),
 ('wt_F_82Z-08624', 0),
 ('wt_M_82Z-CG911', 0),
 ('wt_M_82Z-CH093', 0),
 ('wt_F_82Z-CE775', 0),
 ('wt_M_82Z-09591', 0),
 ('wt_F_82Z-09396', 0),
 ('wt_F_82Z-11888', 0),
 ('wt_F_82Z-09480', 0),
 ('wt_M_82Z-07430', 0),
 ('wt_F_82Z-12142', 0),
 ('wt_F_82Z-07086', 0),
 ('wt_F_82Z-09692', 0),
 ('wt_M_82Z-CG964', 0),
 ('wt_M_82Z-11760', 0),
 ('wt_M_82Z-09590', 0),
 ('wt_F_82Z-CH101', 0),
 ('wt_F_82Z-10394', 0),
 ('wt_M_82Z-11540', 0),
 ('wt_M_82Z-10640', 0),
 ('wt_M_82Z-08477', 0),
 ('wt_M_82Z-07742', 0),
 ('wt_M_82Z-10134', 0),
 ('wt_F_82Z-CE573', 0),
 ('wt_M_82Z-CF161', 0),
 ('wt_M_82Z-09696', 0),
 ('wt_M_82Z-09689', 0),
 ('wt_M_82Z-11305', 0),
 ('wt_M_82Z-11171', 0),
 ('wt_F_82Z-07469', 0),
 ('wt_F_82Z-07714', 0),
 ('wt_F_82Z-0746

In [22]:
len(meshes_dict_copy)

643